In [14]:
import pandas as pd
import numpy as np

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

Q1. Customer Data Encoding and Feature Analysis

(a) Apply One-Hot Encoding to City and Payment_Method

In [16]:
df = pd.DataFrame({"Customer_ID": range(101, 113),
    "City": ["Bhopal", "Delhi", "Mumbai", "Bhopal","Delhi", "Pune", "Mumbai", "Pune",
    "Bhopal", "Delhi", "Pune", "Mumbai"],
    "Payment_Method": ["UPI", "Cash", "Card", "UPI","Card", "Cash", "UPI", "Card","Cash", "UPI", "Card", "Cash"],
    "Purchase_Amount": [500, 1200, 800, 1500, 700, 900, 1800, 1100, 600, 1300, 950, 2000]})

original_shape = df.shape

encoded_df = pd.get_dummies(
    df,
    columns=["City", "Payment_Method"],
    dtype=int)

print(encoded_df)

print("\nOriginal rows:", len(df))
print("Encoded rows:", len(encoded_df))

assert len(df) == len(encoded_df)

new_columns = [col for col in encoded_df.columns
    if col not in df.columns]

print("\nNew columns:", new_columns)
print("Number of new columns:", len(new_columns))


    Customer_ID  Purchase_Amount  City_Bhopal  City_Delhi  City_Mumbai  \
0           101              500            1           0            0   
1           102             1200            0           1            0   
2           103              800            0           0            1   
3           104             1500            1           0            0   
4           105              700            0           1            0   
5           106              900            0           0            0   
6           107             1800            0           0            1   
7           108             1100            0           0            0   
8           109              600            1           0            0   
9           110             1300            0           1            0   
10          111              950            0           0            0   
11          112             2000            0           0            1   

    City_Pune  Payment_Method_Card  P

(b) Compare dimensions and memory usage

In [17]:
print("Original dimensions:", df.shape)
print("Encoded dimensions:", encoded_df.shape)

original_memory = df.memory_usage(deep=True).sum()
encoded_memory = encoded_df.memory_usage(deep=True).sum()

print("\nOriginal memory:", original_memory, "bytes")
print("Encoded memory:", encoded_memory, "bytes")

print("\nMemory difference:",
      encoded_memory - original_memory, "bytes")

print("Original memory in KB:",
      round(original_memory / 1024, 2))

print("Encoded memory in KB:",
      round(encoded_memory / 1024, 2))


Original dimensions: (12, 4)
Encoded dimensions: (12, 9)

Original memory: 1607 bytes
Encoded memory: 996 bytes

Memory difference: -611 bytes
Original memory in KB: 1.57
Encoded memory in KB: 0.97


Q2. One-Hot Encoding in a Machine Learning Pipeline

(a) Build a preprocessing pipeline

In [18]:
df = pd.DataFrame({"Age": [22, 35, 28, 45, 31, 24, 40, 29, 38, 26, 33, 42],
    "City": ["Bhopal", "Delhi", "Mumbai", "Pune","Delhi", "Bhopal", "Mumbai", "Pune",
    "Delhi", "Mumbai", "Bhopal", "Pune"],
    "Browser": ["Chrome", "Firefox", "Safari", "Chrome", "Safari", "Firefox", "Chrome", "Safari",
        "Firefox", "Chrome", "Safari", "Firefox"],
    "Purchase_Amount": [600, 1500, 900, 2200, 1300, 750, 2000, 1100, 1800, 850, 1400, 2100]})

X = df[["Age", "City", "Browser"]]
y = df["Purchase_Amount"]

preprocessor = ColumnTransformer([
    ("city", OneHotEncoder(handle_unknown="ignore"), ["City"]),

    ("browser", OneHotEncoder(
        handle_unknown="ignore"), ["Browser"])], remainder="passthrough")

pipeline = Pipeline([("preprocessing", preprocessor)])

X_encoded = pipeline.fit_transform(X)

feature_names = (pipeline.named_steps["preprocessing"]
    .get_feature_names_out())

print("Feature names:")
print(feature_names)

print("\nDimensions:", X_encoded.shape)

X_array = (X_encoded.toarray()
    if hasattr(X_encoded, "toarray")
    else np.asarray(X_encoded))

print("Only numerical:", np.issubdtype(
    X_array.dtype, np.number))

print("\nEncoded matrix:")
print(X_array)


Feature names:
['city__City_Bhopal' 'city__City_Delhi' 'city__City_Mumbai'
 'city__City_Pune' 'browser__Browser_Chrome' 'browser__Browser_Firefox'
 'browser__Browser_Safari' 'remainder__Age']

Dimensions: (12, 8)
Only numerical: True

Encoded matrix:
[[ 1.  0.  0.  0.  1.  0.  0. 22.]
 [ 0.  1.  0.  0.  0.  1.  0. 35.]
 [ 0.  0.  1.  0.  0.  0.  1. 28.]
 [ 0.  0.  0.  1.  1.  0.  0. 45.]
 [ 0.  1.  0.  0.  0.  0.  1. 31.]
 [ 1.  0.  0.  0.  0.  1.  0. 24.]
 [ 0.  0.  1.  0.  1.  0.  0. 40.]
 [ 0.  0.  0.  1.  0.  0.  1. 29.]
 [ 0.  1.  0.  0.  0.  1.  0. 38.]
 [ 0.  0.  1.  0.  1.  0.  0. 26.]
 [ 1.  0.  0.  0.  0.  0.  1. 33.]
 [ 0.  0.  0.  1.  0.  1.  0. 42.]]


(b) Train the regression model and evaluate MAE

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.25,random_state=42)

model = Pipeline([("preprocessing", ColumnTransformer([("city", OneHotEncoder(handle_unknown="ignore"), ["City"]),

        ("browser", OneHotEncoder(handle_unknown="ignore"), ["Browser"])], remainder="passthrough")),

        ("regressor", RandomForestRegressor(n_estimators=100,random_state=42))])

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)

print("Actual values:", y_test.to_list())
print("Predicted values:", y_pred.round(2))
print("MAE:", round(mae, 2))


Actual values: [1400, 850, 600]
Predicted values: [1160.  1053.5 1026.5]
MAE: 290.0


Q3. High Cardinality and Memory Consumption

(a) Identify high-cardinality columns and compare One-Hot Encoding

In [20]:
df = pd.DataFrame({"Product_Category": ["Electronics", "Clothing", "Books", "Electronics",
        "Food", "Books", "Clothing", "Food"],
    "City": ["Bhopal", "Delhi", "Mumbai", "Pune","Indore", "Delhi", "Bhopal", "Jaipur"],
    "User_ID": ["U101", "U102", "U103", "U104", "U105", "U106", "U107", "U108"],
    "Purchase_Amount": [1200, 800, 400, 2500, 600, 350, 900, 700]})

# Count unique categories
for col in ["Product_Category", "City", "User_ID"]:
    print(col, ":", df[col].nunique())

# One-Hot Encoding
encoded_df = pd.get_dummies(
    df,
    columns=["Product_Category", "City", "User_ID"],
    dtype=int
)

print("\nEncoded DataFrame:")
print(encoded_df)

print("\nOriginal dimensions:", df.shape)
print("Encoded dimensions:", encoded_df.shape)


Product_Category : 4
City : 6
User_ID : 8

Encoded DataFrame:
   Purchase_Amount  Product_Category_Books  Product_Category_Clothing  \
0             1200                       0                          0   
1              800                       0                          1   
2              400                       1                          0   
3             2500                       0                          0   
4              600                       0                          0   
5              350                       1                          0   
6              900                       0                          1   
7              700                       0                          0   

   Product_Category_Electronics  Product_Category_Food  City_Bhopal  \
0                             1                      0            1   
1                             0                      0            0   
2                             0                      0            0

(b) Compare memory consumption and select a better strategy

In [21]:

# Memory usage comparison
original_memory = df.memory_usage(deep=True).sum()
encoded_memory = encoded_df.memory_usage(deep=True).sum()

print("Original memory:", original_memory, "bytes")
print("Encoded memory:", encoded_memory, "bytes")

print("\nOriginal memory in KB:",
      round(original_memory / 1024, 2))

print("Encoded memory in KB:",
      round(encoded_memory / 1024, 2))

print("\nMemory difference:",
      encoded_memory - original_memory, "bytes")

# Better encoding: exclude User_ID
better_df = pd.get_dummies(
    df.drop(columns=["User_ID"]),
    columns=["Product_Category", "City"],
    dtype=int
)

print("\nBetter dimensions:", better_df.shape)

better_memory = better_df.memory_usage(deep=True).sum()
print("Better memory:", better_memory, "bytes")


Original memory: 1504 bytes
Encoded memory: 1348 bytes

Original memory in KB: 1.47
Encoded memory in KB: 1.32

Memory difference: -156 bytes

Better dimensions: (8, 11)
Better memory: 836 bytes
